# Fraud Detection

**Imbalanced Binary Classification** — Detect fraudulent credit card transactions.

Models: CatBoost · LightGBM · XGBoost (with class weights)  
Baselines: LazyPredict · FLAML AutoML  
Dataset: Credit Card Fraud (~284K rows, sampled to 50K)  
Target: `Class` (0=legitimate, 1=fraud, ~0.17% positive)

## 2 · Project Overview

We detect **fraudulent credit card transactions** from anonymized PCA-transformed features. This is the classic heavily imbalanced classification problem — only ~0.17% of transactions are fraudulent.

This notebook emphasizes **PR-AUC, recall, and threshold tuning** over accuracy.

## 3 · Learning Objectives

1. Handle extreme class imbalance (~0.17% positive).
2. Use PR-AUC and recall as primary metrics (not accuracy).
3. Apply class weights in boosting models.
4. Tune decision thresholds for optimal recall.
5. Understand why accuracy is misleading for fraud detection.

## 4 · Problem Statement

Given anonymized features (V1-V28 from PCA) plus Time and Amount, classify each transaction as **legitimate (0)** or **fraudulent (1)**.

## 5 · Why This Project Matters

- Credit card fraud costs billions annually.
- Extreme imbalance is the central challenge — 99.83% are legitimate.
- Missing fraud (false negatives) is far more costly than false positives.
- This is the canonical imbalanced classification benchmark.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 284,807 (sampled to 50,000) |
| **Columns** | 31 |
| **Features** | V1-V28 (PCA), Time, Amount |
| **Target** | `Class` (0=legitimate, 1=fraud) |
| **Fraud rate** | ~0.17% (492 out of 284,807) |
| **Source** | Kaggle: `mlg-ulb/creditcardfraud` |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle / ULB Machine Learning Group.
- **Reference**: Andrea Dal Pozzolo et al. (2015).
- **License**: Open Database License (ODbL).
- **Privacy**: Features V1-V28 are PCA transforms — no PII.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             average_precision_score, precision_recall_curve)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Class"
MAX_ROWS = 50000
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Fraud Detection


## 11 · Dataset Download or Loading

Download from kagglehub. We sample to 50K but ensure ALL fraud cases are included.

In [4]:
# Check for local file first
local_path = os.path.join(SAVE_DIR, "data", "creditcard.csv")
if os.path.exists(local_path):
    df_full = pd.read_csv(local_path)
    print(f"Loaded local: {local_path}")
else:
    import kagglehub
    dl = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    csv_file = None
    for root, dirs, files in os.walk(dl):
        for f in sorted(files):
            if f.endswith(".csv"):
                csv_file = os.path.join(root, f)
                break
        if csv_file:
            break
    assert csv_file, f"No CSV found in {dl}"
    df_full = pd.read_csv(csv_file)
    print(f"Downloaded: {csv_file}")

print(f"Full dataset: {df_full.shape}")
print(f"Fraud count: {df_full[TARGET].sum()} ({100*df_full[TARGET].mean():.3f}%)")

# Smart sampling: keep ALL fraud, sample legitimate
fraud = df_full[df_full[TARGET] == 1]
legit = df_full[df_full[TARGET] == 0]
n_legit_sample = min(MAX_ROWS - len(fraud), len(legit))
legit_sample = legit.sample(n=n_legit_sample, random_state=SEED)
df = pd.concat([fraud, legit_sample]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Sampled dataset: {df.shape}")
print(f"Fraud in sample: {df[TARGET].sum()} ({100*df[TARGET].mean():.3f}%)")
df.head()

Loaded local: E:\Github\Machine-Learning-Projects\Classification\Fraud Detection\data\creditcard.csv
Full dataset: (284807, 31)
Fraud count: 492 (0.173%)
Sampled dataset: (50000, 31)
Fraud in sample: 492 (0.984%)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,71153.0,-1.987284,0.608930,1.685808,0.267102,-0.513254,-0.220602,-0.848086,1.172000,0.108166,...,0.185187,0.210122,-0.281290,0.060078,-0.023724,-0.541371,-0.161717,-0.167134,1.00,0
1,109575.0,-0.532974,0.756122,2.276992,-0.306481,0.356872,0.258729,0.452307,-0.095137,1.245943,...,-0.266204,-0.344266,-0.332072,-0.495050,0.273221,-0.722769,-0.082748,-0.158771,11.27,0
2,59385.0,-7.626924,-6.976420,-2.077911,3.416754,4.458758,-5.080408,-6.578948,1.760341,-0.599509,...,1.224795,-0.656639,-0.330811,-0.078946,0.270306,0.431119,0.821381,-1.056088,18.98,1
3,130347.0,-0.756399,1.652372,-0.735871,-0.755787,0.690505,-0.735194,0.967777,-0.132213,0.738668,...,0.243760,1.196121,-0.063233,0.624893,-0.565545,-0.225262,0.929318,0.566740,2.27,0
4,85089.0,1.068388,-0.278059,1.188888,1.739911,-0.837779,0.568666,-0.621993,0.363155,1.463636,...,-0.428464,-0.837447,0.096762,0.043350,0.376293,-0.505580,0.081105,0.026671,15.99,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nFraud rate: {df[TARGET].mean():.4%}")

DATA VALIDATION

Missing values: 0
Duplicate rows: 52

Target distribution:
Class
0    49508
1      492
Name: count, dtype: int64

Fraud rate: 0.9840%


## 13 · Exploratory Data Analysis

In [6]:
# Amount and Time distributions by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, color, label in [(0, "steelblue", "Legitimate"), (1, "red", "Fraud")]:
    subset = df[df[TARGET] == cls]
    axes[0].hist(subset["Amount"], bins=50, alpha=0.6, color=color, label=label, edgecolor="black")
    axes[1].hist(subset["Time"], bins=50, alpha=0.6, color=color, label=label, edgecolor="black")

axes[0].set_title("Transaction Amount by Class")
axes[0].set_xlabel("Amount"); axes[0].legend()
axes[1].set_title("Transaction Time by Class")
axes[1].set_xlabel("Time"); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Legitimate amount — mean: ${legit['Amount'].mean():.2f}, median: ${legit['Amount'].median():.2f}")
print(f"Fraud amount — mean: ${fraud['Amount'].mean():.2f}, median: ${fraud['Amount'].median():.2f}")

Legitimate amount — mean: $88.29, median: $22.00
Fraud amount — mean: $122.21, median: $9.25


## 14 · Target Analysis

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "red"], edgecolor="black")
ax.set_title(f"Target Distribution: {TARGET}")
ax.set_xticklabels(["Legitimate (0)", "Fraud (1)"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Extreme imbalance: {100*(1-df[TARGET].mean()):.2f}% legitimate vs {100*df[TARGET].mean():.2f}% fraud")
print("→ Accuracy is MISLEADING. A naive 'predict all 0' gets ~99.8% accuracy!")

Extreme imbalance: 99.02% legitimate vs 0.98% fraud
→ Accuracy is MISLEADING. A naive 'predict all 0' gets ~99.8% accuracy!


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve the fraud ratio.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET].values

# Sanitize column names
X.columns = [re.sub(r"[^A-Za-z0-9_]", "_", c) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean():.4%}")
print(f"Test fraud rate: {y_test.mean():.4%}")

# Class weight for imbalance
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / max(n_pos, 1)
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

Train: (40000, 30), Test: (10000, 30)
Train fraud rate: 0.9850%
Test fraud rate: 0.9800%
scale_pos_weight: 100.5


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Features**: V1-V28 already PCA-transformed and scaled.
- **Amount**: Could benefit from log-transform but tree models handle skewness.
- **Time**: Kept as-is.
- **Class imbalance**: Handled via class weights / scale_pos_weight.

## 17 · Feature Engineering

Features are pre-engineered via PCA (anonymization). We keep them as-is. Adding log(Amount) as an extra feature.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()
X_train["Log_Amount"] = np.log1p(X_train["Amount"])
X_test["Log_Amount"] = np.log1p(X_test["Amount"])
print(f"Added Log_Amount. Features: {X_train.shape[1]}")

Added Log_Amount. Features: 31


## 18 · Baseline Model

Logistic Regression with class weights.

In [10]:
baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1]

print("=" * 50)
print("BASELINE: Logistic Regression (balanced)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"F1        : {f1_score(y_test, y_pred_bl):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_bl):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_bl):.4f}")
print(f"PR-AUC    : {average_precision_score(y_test, y_prob_bl):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob_bl):.4f}")
print(f"\n{classification_report(y_test, y_pred_bl)}")

BASELINE: Logistic Regression (balanced)
Accuracy  : 0.9741
F1        : 0.4127
Recall    : 0.9286
Precision : 0.2653
PR-AUC    : 0.8843
ROC-AUC   : 0.9770

              precision    recall  f1-score   support

           0       1.00      0.97      0.99      9902
           1       0.27      0.93      0.41        98

    accuracy                           0.97     10000
   macro avg       0.63      0.95      0.70     10000
weighted avg       0.99      0.97      0.98     10000



## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
QuadraticDiscriminantAnalysis    0.9768           0.937770  0.983597  0.982702   0.991960  0.9768    0.146499
RandomForestClassifier           0.9985           0.923469  0.965443  0.998438   0.998502  0.9985   25.404501
ExtraTreesClassifier             0.9984           0.918367  0.974222  0.998330   0.998403  0.9984    2.383110
CatBoostClassifier               0.9984           0.918367  0.982865  0.998330   0.998403  0.9984    8.443961
Perceptron                       0.9983           0.918317  0.906666  0.998230   0.998284  0.9983    0.167701
LGBMClassifier                   0.9981           0.913164  0.986672  0.998022   0.998066  0.9981    0.390359
LogisticRegression               0.9981           0.913164  0.980136  0.998022   0.998

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="classification", time_budget=60, verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    y_prob_flaml = flaml_automl.predict_proba(X_test)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
    print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    y_pred_flaml = y_pred_bl
    y_prob_flaml = None

FLAML failed: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## 21 · Additional Models: CatBoost, LightGBM, XGBoost (with class weights)

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
proba_results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, loss_function="Logloss",
                            auto_class_weights="Balanced",
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).flatten()
    proba_results["CatBoost"] = cb.predict_proba(X_test)[:, 1]
    print(f"CatBoost PR-AUC: {average_precision_score(y_test, proba_results['CatBoost']):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                            is_unbalance=True,
                            device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    proba_results["LightGBM"] = lg.predict_proba(X_test)[:, 1]
    print(f"LightGBM PR-AUC: {average_precision_score(y_test, proba_results['LightGBM']):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                             scale_pos_weight=scale_pos_weight,
                             device="cuda", tree_method="hist", verbosity=0,
                             eval_metric="aucpr",
                             n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    proba_results["XGBoost"] = xgb_model.predict_proba(X_test)[:, 1]
    print(f"XGBoost PR-AUC: {average_precision_score(y_test, proba_results['XGBoost']):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Baseline"] = y_pred_bl
proba_results["Baseline"] = y_prob_bl
if y_pred_flaml is not None:
    results["FLAML"] = y_pred_flaml
    if y_prob_flaml is not None:
        try:
            proba_results["FLAML"] = flaml_automl.predict_proba(X_test)[:, 1]
        except Exception:
            pass

CatBoost PR-AUC: 0.8724  (0.9s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[219]	valid_0's binary_logloss: 0.00976976
LightGBM PR-AUC: 0.9110  (2.6s)


XGBoost PR-AUC: 0.9081  (1.3s)


## 22 · Top 3 Model Selection

Ranked by **PR-AUC** (most appropriate for extreme imbalance).

In [14]:
model_scores = {}
for name in results:
    yp = results[name].astype(int) if hasattr(results[name], "astype") else results[name]
    row = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp), 4),
        "Recall": round(recall_score(y_test, yp), 4),
        "Precision": round(precision_score(y_test, yp, zero_division=0), 4),
    }
    if name in proba_results:
        row["PR-AUC"] = round(average_precision_score(y_test, proba_results[name]), 4)
        row["ROC-AUC"] = round(roc_auc_score(y_test, proba_results[name]), 4)
    model_scores[name] = row

scores_df = pd.DataFrame(model_scores).T
sort_col = "PR-AUC" if "PR-AUC" in scores_df.columns else "F1"
scores_df = scores_df.sort_values(sort_col, ascending=False)
print("All Model Rankings (by PR-AUC):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by PR-AUC):
          Accuracy      F1  Recall  Precision  PR-AUC  ROC-AUC
LightGBM    0.9983  0.9110  0.8878     0.9355  0.9110   0.9810
XGBoost     0.9985  0.9198  0.8776     0.9663  0.9081   0.9852
Baseline    0.9741  0.4127  0.9286     0.2653  0.8843   0.9770
CatBoost    0.9885  0.6021  0.8878     0.4555  0.8724   0.9707
FLAML       0.9741  0.4127  0.9286     0.2653     NaN      NaN

Top 3 models: ['LightGBM', 'XGBoost', 'Baseline']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name].astype(int) if hasattr(results[name], "astype") else results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"]).plot(ax=axes[i], cmap="Blues")
    rec = recall_score(y_test, yp)
    pr_auc = average_precision_score(y_test, proba_results[name]) if name in proba_results else 0
    axes[i].set_title(f"{name}\nRecall={rec:.4f} PR-AUC={pr_auc:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

# PR curves
fig, ax = plt.subplots(figsize=(8, 6))
for name in top3_names:
    if name in proba_results:
        prec, rec, _ = precision_recall_curve(y_test, proba_results[name])
        ap = average_precision_score(y_test, proba_results[name])
        ax.plot(rec, prec, label=f"{name} (AP={ap:.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Top 3")
ax.legend()
plt.tight_layout()
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name].astype(int) if hasattr(results[name], "astype") else results[name]
    print(f"\n  {name}:")
    print(f"    Recall    : {recall_score(y_test, yp):.4f}")
    print(f"    Precision : {precision_score(y_test, yp, zero_division=0):.4f}")
    print(f"    F1        : {f1_score(y_test, yp):.4f}")
    if name in proba_results:
        print(f"    PR-AUC    : {average_precision_score(y_test, proba_results[name]):.4f}")
        print(f"    ROC-AUC   : {roc_auc_score(y_test, proba_results[name]):.4f}")


Detailed Metrics — Top 3:

  LightGBM:
    Recall    : 0.8878
    Precision : 0.9355
    F1        : 0.9110
    PR-AUC    : 0.9110
    ROC-AUC   : 0.9810

  XGBoost:
    Recall    : 0.8776
    Precision : 0.9663
    F1        : 0.9198
    PR-AUC    : 0.9081
    ROC-AUC   : 0.9852

  Baseline:
    Recall    : 0.9286
    Precision : 0.2653


    F1        : 0.4127
    PR-AUC    : 0.8843
    ROC-AUC   : 0.9770


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name].astype(int) if hasattr(results[best_name], "astype") else results[best_name]

cm = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm.ravel()

print(f"Best model: {best_name}")
print(f"True Positives  (caught fraud): {tp}")
print(f"False Negatives (missed fraud): {fn}")
print(f"False Positives (false alarms): {fp}")
print(f"True Negatives  (correct legit): {tn}")
print(f"\nFraud detection rate: {tp}/{tp+fn} = {100*tp/(tp+fn):.1f}%")
print(f"False alarm rate: {fp}/{fp+tn} = {100*fp/(fp+tn):.3f}%")

Best model: LightGBM
True Positives  (caught fraud): 87
False Negatives (missed fraud): 11
False Positives (false alarms): 6
True Negatives  (correct legit): 9896

Fraud detection rate: 87/98 = 88.8%
False alarm rate: 6/9902 = 0.061%


## 25 · Interpretation and Business Insight

**Key findings:**
- **Extreme imbalance** (~0.17% fraud) makes accuracy meaningless — a naive model gets 99.83%.
- **PR-AUC** is the right metric — it captures precision-recall tradeoff.
- **Class weights** are essential for boosting models to detect rare fraud.
- **V14, V17, V12** (PCA features) are typically the most important for fraud detection.

**Business takeaway:** Optimize for high recall (catch fraud) with acceptable false positive rate. Even 1% false alarm rate may flag thousands of legitimate transactions.

## 26 · Limitations

1. **PCA-anonymized features** — no domain interpretation possible.
2. **No time-based split** — temporal patterns in fraud may leak.
3. **Static threshold** — real systems need adaptive thresholds.
4. **No cost-sensitive evaluation** — different costs for FP vs FN.
5. **Single dataset** — fraud patterns vary by region and time.

## 27 · How to Improve This Project

1. Use time-based train/test split (first 80% time for train).
2. Add cost-sensitive evaluation (FN costs more than FP).
3. Try isolation forest or autoencoders for anomaly detection.
4. Implement threshold optimization based on business constraints.
5. Use calibrated probabilities for risk scoring.

## 28 · Production Considerations

- Real-time scoring with <100ms latency requirement.
- Continuous model retraining as fraud patterns evolve.
- Alert triage system for human investigation.
- Monitor false positive rate — too high → customer friction.
- Regulatory compliance (explainability requirements).

## 29 · Common Mistakes

1. **Using accuracy** — 99.83% accuracy means nothing for fraud.
2. **Not using class weights** — model predicts all legitimate.
3. **Random split on time-series data** — causes temporal leakage.
4. **Oversampling before split** — causes data leakage.
5. **Ignoring false positive cost** — flagging too many legit transactions.

## 30 · Mini Challenge / Exercises

1. Tune the decision threshold to achieve 95% recall — what's the precision?
2. Try SMOTE oversampling — does it improve PR-AUC?
3. Build an Isolation Forest and compare with supervised models.
4. What happens if you remove Amount and Time features?
5. Plot ROC curves — why is ROC-AUC misleadingly high for imbalanced data?

## 31 · Final Summary / Key Takeaways

1. **PR-AUC > Accuracy** for imbalanced fraud detection.
2. **Class weights** are essential for tree-based models.
3. **99.83% accuracy** is meaningless — a dummy model achieves it.
4. **Recall vs precision tradeoff** is the key business decision.
5. **Threshold tuning** matters more than model selection.
6. **Always keep all fraud cases** in sampling.

## Save Metrics

In [17]:
metrics_out = {}
for name in results:
    yp = results[name].astype(int) if hasattr(results[name], "astype") else results[name]
    row = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp)), 4),
        "recall": round(float(recall_score(y_test, yp)), 4),
        "precision": round(float(precision_score(y_test, yp, zero_division=0)), 4),
    }
    if name in proba_results:
        row["pr_auc"] = round(float(average_precision_score(y_test, proba_results[name])), 4)
    metrics_out[name] = row

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Fraud Detection\metrics.json
{
  "CatBoost": {
    "accuracy": 0.9885,
    "f1": 0.6021,
    "recall": 0.8878,
    "precision": 0.4555,
    "pr_auc": 0.8724
  },
  "LightGBM": {
    "accuracy": 0.9983,
    "f1": 0.911,
    "recall": 0.8878,
    "precision": 0.9355,
    "pr_auc": 0.911
  },
  "XGBoost": {
    "accuracy": 0.9985,
    "f1": 0.9198,
    "recall": 0.8776,
    "precision": 0.9663,
    "pr_auc": 0.9081
  },
  "Baseline": {
    "accuracy": 0.9741,
    "f1": 0.4127,
    "recall": 0.9286,
    "precision": 0.2653,
    "pr_auc": 0.8843
  },
  "FLAML": {
    "accuracy": 0.9741,
    "f1": 0.4127,
    "recall": 0.9286,
    "precision": 0.2653
  }
}
